In [ ]:
!pip install -q MDAnalysis MDTraj torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
output_dir = '/content/drive/MyDrive/PATH/'
os.makedirs(output_dir, exist_ok=True)

print(f"Working directory created at: {output_dir}")

Working directory created at: /content/drive/MyDrive/ABE_RF_Diffusion/Two_class_sparse_1/PAPER_FINAL/


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
import MDAnalysis as mda
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# Load the data_frames list from the file
loaded_data_frames = torch.load(os.path.join(output_dir, 'GAT_Two_Class_data.pt'), weights_only=False)

In [ ]:
# Verify the loaded data
print(loaded_data_frames[0])

Data(x=[552, 3], edge_index=[2, 9076], edge_attr=[552, 552], y=[1])


In [ ]:
# Initialize lists to store train and test frames
train_frames, test_frames = [], []

# Define test frame indices:
# For label 0: take frames 99, 199, 299, ... 999 (10 frames)
test_indices_label0 = list(range(1, 1000, 10))
# For label 1: frames are from index 1000 to 1999, so take frames 1099, 1199, ... 1999 (10 frames)
test_indices_label1 = list(range(1001, 2000, 10))

# Combine test indices from both labels
test_indices = test_indices_label0 + test_indices_label1

test_frames = [loaded_data_frames[i] for i in test_indices]
train_frames = [data for i, data in enumerate(loaded_data_frames) if i not in test_indices]

# Check test set size for each label
test_label_0_count = sum([data.y.item() == 0 for data in test_frames])
test_label_1_count = sum([data.y.item() == 1 for data in test_frames])
print(f"Test set size: Label 0: {test_label_0_count}, Label 1: {test_label_1_count}")

Test set size: Label 0: 100, Label 1: 100


In [ ]:
class GATNet(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GATNet, self).__init__()
        self.conv1 = GATConv(in_channels, 8, heads=8, concat=True)
        self.conv2 = GATConv(8 * 8, 16, heads=8, concat=False)
        self.lin = torch.nn.Linear(16, out_channels)

    def forward(self, x, edge_index, edge_weight=None, batch=None):
        x, attn_weights_1 = self.conv1(x, edge_index, return_attention_weights=True)
        x = F.relu(x)
        x, attn_weights_2 = self.conv2(x, edge_index, return_attention_weights=True)
        x = global_mean_pool(x, torch.zeros(x.size(0), dtype=torch.long))
        x = self.lin(x)
        return F.log_softmax(x, dim=1), attn_weights_1, attn_weights_2

In [ ]:
# Update model for two classes (out_channels=2)
model = GATNet(in_channels=3, out_channels=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.CrossEntropyLoss()

train_loader = DataLoader(train_frames, batch_size=1, shuffle=True)

def train_model(loader, model, optimizer, loss_fn, epochs=100, patience=10):
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for data in loader:
            optimizer.zero_grad()
            output, attn_weights_1, attn_weights_2 = model(data.x, data.edge_index)
            loss = loss_fn(output, data.y.view(-1))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        print(f'Epoch {epoch}, Loss: {avg_loss}')

        # Early stopping and model saving
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(output_dir, 'GAT_Two_Class_model.pt'))
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

# Train the model
train_model(train_loader, model, optimizer, loss_fn)

/tmp/ipython-input-10273822.py:6: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_frames, batch_size=1, shuffle=True)


Epoch 0, Loss: 0.3115054999095575
Epoch 1, Loss: 0.01631763231922763
Epoch 2, Loss: 0.00236965032676438
Epoch 3, Loss: 0.01610793540390591
Epoch 4, Loss: 0.0076214072509261
Epoch 5, Loss: 0.0007353330790168992
Epoch 6, Loss: 0.0005350877403560419
Epoch 7, Loss: 0.00834679952044129
Epoch 8, Loss: 0.010479260232914196
Epoch 9, Loss: 0.0003871154464149345
Epoch 10, Loss: 0.00026515235462943015
Epoch 11, Loss: 0.011790327102014062
Epoch 12, Loss: 0.0068100309196010515
Epoch 13, Loss: 0.0005194111818806688
Epoch 14, Loss: 8.833960310100666e-05
Epoch 15, Loss: 7.607816973405019e-05
Epoch 16, Loss: 0.01037388877521551
Epoch 17, Loss: 0.00015815655177577682
Epoch 18, Loss: 0.006716882686716809
Epoch 19, Loss: 0.0006223936232146185
Epoch 20, Loss: 0.002118307003934454
Epoch 21, Loss: 5.421421246510979e-05
Epoch 22, Loss: 0.008249947655799726
Epoch 23, Loss: 5.899705247963756e-05
Epoch 24, Loss: 4.6145553384238586e-05
Epoch 25, Loss: 0.007669541515527156
Epoch 26, Loss: 6.003069195629725e-05
Epo

In [ ]:
# Evaluate on the test set
y_true, y_pred = [], []

for data in test_frames:
    output, _, _ = model(data.x, data.edge_index)
    y_true.append(data.y.item())
    y_pred.append(output.argmax(dim=1).item())

accuracy = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 100.00%


In [ ]:
# Confusion matrix and plot for two classes
conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, cmap="Blues", fmt="d",
            xticklabels=[0, 1], yticklabels=[0, 1])
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.title('Confusion Matrix for Test Set')

# Save the confusion matrix plot as a PNG file
output_path = os.path.join(output_dir, 'confusion_matrix.png')
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Confusion matrix saved at: {output_path}")

Confusion matrix saved at: /content/drive/MyDrive/ABE_RF_Diffusion/Two_class_sparse_1/PAPER_FINAL/confusion_matrix.png
